# QMProt Dataset Analysis with Bokeh

This notebook provides comprehensive visualizations of the QMProt molecular dataset using Bokeh interactive plots.

## Dataset Overview
The QMProt dataset contains molecular information for amino acids and small molecules, including:
- **Test molecules**: H2, H2O, Glutamine
- **Amino acids**: Glycine, Alanine, Histidine, Leucine, Isoleucine, Tryptophan, Phenylalanine, Tyrosine, Serine, Valine

Each molecule includes quantum chemical properties such as number of atoms, electrons, orbitals, qubits, and Hamiltonian coefficients.

In [15]:
!pip install bokeh --quiet
# Import required libraries
import json
import numpy as np
import pandas as pd
from bokeh.plotting import figure, show, output_notebook, output_file
from bokeh.layouts import column, row, gridplot
from bokeh.models import HoverTool, ColumnDataSource, Div, TableColumn, DataTable
from bokeh.palettes import Category20, Viridis256
from bokeh.transform import factor_cmap
from bokeh.io import curdoc, export_png, export_svgs
import warnings
warnings.filterwarnings('ignore')
import os
from pathlib import Path

# Enable Bokeh output in notebook
output_notebook()

# Setup plot export directory
plots_dir = Path("qmprot_overview_plots")
plots_dir.mkdir(exist_ok=True)
print(f"📁 Created plots directory: {plots_dir.absolute()}")

# Function to save plots
def save_plot(plot, filename):
    """Save a plot to both HTML and PNG (if possible)"""
    html_path = plots_dir / f"{filename}.html"
    
    # Save as HTML (always works)
    output_file(str(html_path))
    show(plot)  # This saves the HTML file
    print(f"💾 Saved: {html_path}")
    
    # Try to save as PNG (requires additional setup)
    try:
        png_path = plots_dir / f"{filename}.png"
        export_png(plot, filename=str(png_path))
        print(f"🖼️ Saved: {png_path}")
    except Exception as e:
        print(f"ℹ️  PNG export failed (install selenium + chromedriver for PNG): {e}")

print("✅ Libraries imported successfully!")
print("📊 Bokeh notebook output enabled")
print("💾 Plot export functionality ready")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Loading BokehJS ...

📁 Created plots directory: /Users/sanskriti/Documents/GitHub/qmprot_gse/explore_init/simple_plots/qmprot_overview_plots
✅ Libraries imported successfully!
📊 Bokeh notebook output enabled
💾 Plot export functionality ready


In [16]:
# Load and process QMProt dataset
json_path = "../../framework/data/qmprot.json"

with open(json_path, 'r') as f:
    data = json.load(f)

# Extract molecules from both categories
molecules = []
molecules.extend([(mol, 'Test Molecule') for mol in data['test_molecules']])
molecules.extend([(mol, 'Amino Acid') for mol in data['amino_acids']])

# Create DataFrame for analysis
df_data = []
for mol, category in molecules:
    df_data.append({
        'name': mol['name'],
        'abbreviation': mol['abbreviation'],
        'category': category,
        'molecular_formula': mol['mf'],
        'n_atoms': mol['n_atoms'],
        'n_electrons': mol['n_electrons'],
        'n_orbitals': mol['n_orbitals'],
        'n_qubits': mol['n_qubits'],
        'n_coefficients': mol['n_coefficients'],
        'energy': mol['energy'],
        'basis': mol.get('basis', 'sto-3g')
    })

df = pd.DataFrame(df_data)

# Display basic statistics
print(f" Dataset Statistics:")
print(f"   Total molecules: {len(df)}")
print(f"   Test molecules: {len(df[df['category'] == 'Test Molecule'])}")
print(f"   Amino acids: {len(df[df['category'] == 'Amino Acid'])}")
print(f"\n Data ranges:")
for col in ['n_atoms', 'n_electrons', 'n_orbitals', 'n_qubits', 'n_coefficients']:
    print(f"   {col}: {df[col].min():.0f} - {df[col].max():.0f}")
print(f"   Energy: {df['energy'].min():.2f} - {df['energy'].max():.2f} Hartree")

# Show first few rows
print(f"\n📋 Sample data:")
display(df.head())

 Dataset Statistics:
   Total molecules: 13
   Test molecules: 3
   Amino acids: 10

 Data ranges:
   n_atoms: 2 - 27
   n_electrons: 2 - 98
   n_orbitals: 2 - 80
   n_qubits: 4 - 160
   n_coefficients: 15 - 42500000
   Energy: -672.12 - -1.14 Hartree

📋 Sample data:


,name,abbreviation,category,molecular_formula,n_atoms,n_electrons,n_orbitals,n_qubits,n_coefficients,energy,basis
0,glutamine,gln,Test Molecule,C5H10N2O3,20,10,8,16,5793,-521.750624,sto-3g
1,hydrogen,h2,Test Molecule,H2,2,2,2,4,15,-1.137306,sto-3g
2,water,h2o,Test Molecule,H2O,3,10,7,14,60,-75.584140,sto-3g
3,glycine,gly,Amino Acid,C2H5NO2,10,40,30,60,1500000,-282.867420,sto-3g
4,alanine,ala,Amino Acid,C3H7NO2,13,48,39,78,4200000,-321.752310,sto-3g


In [17]:
# Create interactive bar charts for molecular properties
def create_property_chart(property_name, title, log_scale=False):
    """Create an interactive bar chart for a molecular property"""
    
    # Prepare data
    source = ColumnDataSource(df)
    colors = factor_cmap('category', palette=Category20[3][:2], factors=df['category'].unique())
    
    # Create figure with log scale option
    if log_scale:
        p = figure(
            x_range=df['abbreviation'], 
            height=400, 
            width=800,
            title=title,
            toolbar_location="above",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            y_axis_type="log"  # Set log scale at figure creation
        )
    else:
        p = figure(
            x_range=df['abbreviation'], 
            height=400, 
            width=800,
            title=title,
            toolbar_location="above",
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )
    
    # Add bars
    bars = p.vbar(
        x='abbreviation', 
        top=property_name, 
        width=0.8,
        source=source,
        color=colors,
        alpha=0.8
    )
    
    # Configure hover tool
    hover = HoverTool(tooltips=[
        ("Molecule", "@name (@abbreviation)"),
        ("Category", "@category"),
        ("Formula", "@molecular_formula"),
        (title.split(':')[0], f"@{property_name}{{0,0}}"),
        ("Energy", "@energy{0.000} Ha")
    ])
    p.add_tools(hover)
    
    # Styling
    p.xgrid.grid_line_color = None
    p.xaxis.axis_label = "Molecule"
    p.yaxis.axis_label = title.split(':')[0]
    p.xaxis.major_label_orientation = 45
    
    return p

# Create property distribution charts
p1 = create_property_chart('n_atoms', 'Number of Atoms: Molecular Size Distribution')
p2 = create_property_chart('n_electrons', 'Number of Electrons: Electronic Complexity')
p3 = create_property_chart('n_orbitals', 'Number of Orbitals: Basis Set Size')
p4 = create_property_chart('n_qubits', 'Number of Qubits: Quantum Circuit Size')

# Create layout
grid = gridplot([[p1, p2], [p3, p4]], sizing_mode='stretch_width')

# Show and save
show(grid)
save_plot(grid, "01_molecular_properties_grid")

💾 Saved: qmprot_overview_plots/01_molecular_properties_grid.html
ℹ️  PNG export failed (install selenium + chromedriver for PNG): To use bokeh.io image export functions you need selenium ('conda install selenium' or 'pip install selenium')


In [18]:
# Hamiltonian coefficients analysis (log scale due to large range)
p5 = create_property_chart('n_coefficients', 'Number of Hamiltonian Coefficients: Computational Complexity', log_scale=True)

# Add gridlines and formatting for log scale plot
p5.ygrid.minor_grid_line_color = 'gray'
p5.ygrid.minor_grid_line_alpha = 0.3

# Show and save
show(p5)
save_plot(p5, "02_hamiltonian_coefficients")

💾 Saved: qmprot_overview_plots/02_hamiltonian_coefficients.html
ℹ️  PNG export failed (install selenium + chromedriver for PNG): To use bokeh.io image export functions you need selenium ('conda install selenium' or 'pip install selenium')


In [19]:
# Energy distribution histogram
def create_energy_histogram():
    """Create an interactive histogram for energy distribution"""
    
    # Calculate histogram data
    hist, edges = np.histogram(df['energy'], bins=15)
    
    # Create data source
    hist_df = pd.DataFrame({
        'hist': hist,
        'left': edges[:-1],
        'right': edges[1:],
        'center': (edges[:-1] + edges[1:]) / 2
    })
    source = ColumnDataSource(hist_df)
    
    # Create figure
    p = figure(
        height=400, 
        width=800,
        title="Energy Distribution: Ground State Energies (Hartree)",
        toolbar_location="above",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    # Add histogram bars
    bars = p.quad(
        top='hist', 
        bottom=0, 
        left='left', 
        right='right',
        source=source,
        fill_color='steelblue',
        line_color='white',
        alpha=0.8
    )
    
    # Add hover tool
    hover = HoverTool(tooltips=[
        ("Energy Range", "@left{0.0} to @right{0.0} Ha"),
        ("Count", "@hist molecules"),
        ("Bin Center", "@center{0.00} Ha")
    ])
    p.add_tools(hover)
    
    # Styling
    p.xaxis.axis_label = "Energy (Hartree)"
    p.yaxis.axis_label = "Number of Molecules"
    p.grid.grid_line_alpha = 0.3
    
    return p

# Create and show energy histogram
energy_hist = create_energy_histogram()
show(energy_hist)
save_plot(energy_hist, "03_energy_distribution")

💾 Saved: qmprot_overview_plots/03_energy_distribution.html
ℹ️  PNG export failed (install selenium + chromedriver for PNG): To use bokeh.io image export functions you need selenium ('conda install selenium' or 'pip install selenium')


In [20]:
# Correlation analysis: Scatter plots
def create_correlation_plot(x_prop, y_prop, title):
    """Create scatter plot showing correlation between two properties"""
    
    source = ColumnDataSource(df)
    colors = factor_cmap('category', palette=Category20[3][:2], factors=df['category'].unique())
    
    p = figure(
        height=400, 
        width=600,
        title=title,
        toolbar_location="above",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    # Add scatter points
    circles = p.circle(
        x=x_prop, 
        y=y_prop, 
        source=source,
        size=12,
        color=colors,
        alpha=0.7,
        line_color='white',
        line_width=1
    )
    
    # Hover tool
    hover = HoverTool(tooltips=[
        ("Molecule", "@name (@abbreviation)"),
        ("Category", "@category"),
        ("Formula", "@molecular_formula"),
        (x_prop.replace('_', ' ').title(), f"@{x_prop}{{0,0}}"),
        (y_prop.replace('_', ' ').title(), f"@{y_prop}{{0,0}}")
    ])
    p.add_tools(hover)
    
    # Styling
    p.xaxis.axis_label = x_prop.replace('_', ' ').title()
    p.yaxis.axis_label = y_prop.replace('_', ' ').title()
    
    return p

# Create correlation plots
corr1 = create_correlation_plot('n_atoms', 'n_electrons', 'Atoms vs Electrons: Molecular Size Relationship')
corr2 = create_correlation_plot('n_qubits', 'n_coefficients', 'Qubits vs Coefficients: Computational Scaling')

# Layout correlation plots
corr_layout = row(corr1, corr2, sizing_mode='stretch_width')
show(corr_layout)
save_plot(corr_layout, "04_correlation_analysis")

💾 Saved: qmprot_overview_plots/04_correlation_analysis.html
ℹ️  PNG export failed (install selenium + chromedriver for PNG): To use bokeh.io image export functions you need selenium ('conda install selenium' or 'pip install selenium')


In [21]:
# Computational complexity analysis
def create_complexity_heatmap():
    """Create a heatmap showing computational complexity relationships"""
    
    # Calculate complexity metrics
    df_complexity = df.copy()
    df_complexity['qubits_per_atom'] = df_complexity['n_qubits'] / df_complexity['n_atoms']
    df_complexity['coeffs_per_qubit'] = df_complexity['n_coefficients'] / df_complexity['n_qubits']
    df_complexity['log_coeffs'] = np.log10(df_complexity['n_coefficients'])
    
    # Calculate circle sizes based on coeffs_per_qubit
    max_coeffs_per_qubit = df_complexity['coeffs_per_qubit'].max()
    df_complexity['circle_size'] = 8 + (df_complexity['coeffs_per_qubit'] / max_coeffs_per_qubit) * 25
    
    # Create figure
    p = figure(
        x_range=df['abbreviation'],
        height=400,
        width=800,
        title="Computational Complexity: Qubits per Atom vs Coefficients per Qubit",
        toolbar_location="above",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    # Map colors to log coefficients
    mapper = linear_cmap(field_name='log_coeffs', palette=Viridis256, 
                         low=df_complexity['log_coeffs'].min(), 
                         high=df_complexity['log_coeffs'].max())
    
    # Add circles with pre-calculated sizes and color mapping
    source = ColumnDataSource(df_complexity)
    circles = p.circle(
        x='abbreviation',
        y='qubits_per_atom',
        source=source,
        size='circle_size',  # Use pre-calculated size column
        color=mapper,
        alpha=0.8,
        line_color='white',
        line_width=1
    )
    
    # Hover tool
    hover = HoverTool(tooltips=[
        ("Molecule", "@name (@abbreviation)"),
        ("Category", "@category"),
        ("Qubits per Atom", "@qubits_per_atom{0.0}"),
        ("Coefficients per Qubit", "@coeffs_per_qubit{0,0}"),
        ("Total Coefficients", "@n_coefficients{0,0}")
    ])
    p.add_tools(hover)
    
    # Styling
    p.xaxis.axis_label = "Molecule"
    p.yaxis.axis_label = "Qubits per Atom"
    p.xaxis.major_label_orientation = 45
    
    return p

from bokeh.transform import linear_cmap

complexity_plot = create_complexity_heatmap()
show(complexity_plot)
save_plot(complexity_plot, "05_computational_complexity")

💾 Saved: qmprot_overview_plots/05_computational_complexity.html
ℹ️  PNG export failed (install selenium + chromedriver for PNG): To use bokeh.io image export functions you need selenium ('conda install selenium' or 'pip install selenium')


In [22]:
# Summary statistics table
def create_summary_table():
    """Create an interactive table with summary statistics"""
    
    # Calculate statistics by category
    stats_data = []
    for category in df['category'].unique():
        cat_data = df[df['category'] == category]
        stats_data.append({
            'Category': category,
            'Count': len(cat_data),
            'Avg Atoms': f"{cat_data['n_atoms'].mean():.1f}",
            'Avg Electrons': f"{cat_data['n_electrons'].mean():.1f}",
            'Avg Qubits': f"{cat_data['n_qubits'].mean():.1f}",
            'Avg Coefficients': f"{cat_data['n_coefficients'].mean():.0f}",
            'Min Energy (Ha)': f"{cat_data['energy'].min():.2f}",
            'Max Energy (Ha)': f"{cat_data['energy'].max():.2f}"
        })
    
    stats_df = pd.DataFrame(stats_data)
    source = ColumnDataSource(stats_df)
    
    columns = [
        TableColumn(field="Category", title="Category"),
        TableColumn(field="Count", title="Count"),
        TableColumn(field="Avg Atoms", title="Avg Atoms"),
        TableColumn(field="Avg Electrons", title="Avg Electrons"),
        TableColumn(field="Avg Qubits", title="Avg Qubits"),
        TableColumn(field="Avg Coefficients", title="Avg Coefficients"),
        TableColumn(field="Min Energy (Ha)", title="Min Energy (Ha)"),
        TableColumn(field="Max Energy (Ha)", title="Max Energy (Ha)")
    ]
    
    table = DataTable(source=source, columns=columns, width=800, height=200)
    
    return table

# Create summary section
summary_title = Div(text="<h2>📊 Dataset Summary Statistics</h2>", style={'font-size': '20px'})
summary_table = create_summary_table()

# Combine title and table
summary_layout = column(summary_title, summary_table)

# Show summary
show(summary_layout)
save_plot(summary_layout, "06_summary_statistics")

# Create a final summary message about exported plots
print("\n🎉 All plots have been generated and saved!")
print(f"📂 Plots saved to: {plots_dir.absolute()}")
print("\n📋 Generated files:")
plot_files = [
    "01_molecular_properties_grid.html",
    "02_hamiltonian_coefficients.html", 
    "03_energy_distribution.html",
    "04_correlation_analysis.html",
    "05_computational_complexity.html",
    "06_summary_statistics.html"
]
for i, filename in enumerate(plot_files, 1):
    print(f"  {i}. {filename}")
print("\n💡 HTML files can be opened in any web browser for interactive viewing!")

AttributeError: unexpected attribute 'style' to Div, similar attributes are styles, stylesheets or syncable

## 🔍 Key Insights from the QMProt Dataset

### Molecular Diversity
- **Size range**: From H2 (2 atoms) to Tryptophan (27 atoms)
- **Electronic complexity**: 2 electrons (H2) to 98 electrons (Tryptophan)
- **Computational scaling**: 4 qubits (H2) to 160 qubits (Tryptophan)

### Computational Challenges
- **Hamiltonian coefficients** vary dramatically (15 to 42.5M terms)
- **Logarithmic scaling** in computational complexity
- **Memory requirements** scale with number of coefficients

### Energy Patterns
- **Ground state energies** span a wide range due to molecular size
- **Binding energy** generally correlates with molecular size
- **Test molecules** (H2, H2O) have simpler electronic structures

### VQE Implications
- **Small molecules** (H2, H2O) ideal for algorithm testing
- **Amino acids** require truncation strategies for NISQ devices
- **Computational resources** scale exponentially with system size

This analysis helps guide:
1. **Algorithm selection** based on molecular complexity
2. **Truncation strategies** for large systems
3. **Benchmarking approaches** using molecules of different scales